# Parallel maximum entropy knockoffs

This notebook compares the existing `method=:maxent` solver with the experimental `method=:maxent_fast` solver on AR(1) Gaussian model-X data with `p = 1000` features. For an actual four-thread timing run, start Julia/Jupyter with `JULIA_NUM_THREADS=4` before opening the notebook.

In [1]:
using Knockoffs
using LinearAlgebra
using Random
using Statistics
using Distributions

BLAS.set_num_threads(1)
Threads.nthreads()

4

## Simulate AR(1) model-X data

In [2]:
Random.seed!(2026)

n = 500
p = 1000
k = 60
q = 0.10
ρ = 0.4

Σ = [ρ^abs(i - j) for i in 1:p, j in 1:p]
μ = zeros(p)
X = rand(MvNormal(μ, Symmetric(Σ)), n)' |> Matrix

causal = sort!(shuffle(1:p)[1:k])
β = zeros(p)
β[causal] .= rand([-1.0, 1.0], k) .* 2.5
y = X * β + randn(n);

In [3]:
function run_knockoff_trial(method::Symbol; solver_kwargs...)
    Random.seed!(2027)
    solve_time = @elapsed s = solve_s(Symmetric(Σ), method; solver_kwargs...)
    min_eig = eigmin(Symmetric(2Σ - Diagonal(s)))

    Random.seed!(2028)
    ko_time = @elapsed ko = modelX_gaussian_knockoffs(X, method, μ, Σ; solver_kwargs...)

    Random.seed!(2029)
    fit_time = @elapsed fit = fit_lasso(y, ko; filter_method=:knockoff, debias=nothing)
    selected = select_variables(fit, q)
    true_selected = intersect(selected, causal)
    false_selected = setdiff(selected, causal)
    fdp = isempty(selected) ? 0.0 : length(false_selected) / length(selected)
    power = length(true_selected) / length(causal)

    return (; method, solve_time, ko_time, fit_time, min_eig,
        selected=length(selected), fdp, power, mean_s=mean(s))
end

function print_result(r)
    println("method:     ", r.method)
    println("solve time: ", round(r.solve_time, digits=3), " seconds")
    println("ko time:    ", round(r.ko_time, digits=3), " seconds")
    println("fit time:   ", round(r.fit_time, digits=3), " seconds")
    println("min eig:    ", r.min_eig)
    println("selected:   ", r.selected)
    println("FDP:        ", round(r.fdp, digits=3))
    println("power:      ", round(r.power, digits=3))
    println("mean(s):    ", round(r.mean_s, digits=3))
end;

## Baseline: `method=:maxent`

In [4]:
maxent_result = run_knockoff_trial(:maxent; niter=100, tol=1e-5)
print_result(maxent_result)

method:     maxent
solve time: 4.383 seconds
ko time:    2.34 seconds
fit time:   2.789 seconds
min eig:    0.24654608383770443
selected:   66
FDP:        0.091
power:      1.0
mean(s):    0.611


## Experimental: `method=:maxent_fast`

In [5]:
fast_result = run_knockoff_trial(:maxent_fast;
    niter=100,
    tol=1e-5,
    nworkers=4,
    min_spacing=250,
    shuffle_offsets=true,
    parallel_cholesky_updates=true)
print_result(fast_result)

method:     maxent_fast
solve time: 0.987 seconds
ko time:    0.898 seconds
fit time:   1.444 seconds
min eig:    0.24654555346759
selected:   64
FDP:        0.062
power:      1.0
mean(s):    0.611


## Speed and FDR check

In [6]:
println("solve_s speedup: ", round(maxent_result.solve_time / fast_result.solve_time, digits=2), "x")
println("baseline FDP:    ", round(maxent_result.fdp, digits=3))
println("fast FDP:        ", round(fast_result.fdp, digits=3))
println("baseline power:  ", round(maxent_result.power, digits=3))
println("fast power:      ", round(fast_result.power, digits=3))
println("baseline min eig:", maxent_result.min_eig)
println("fast min eig:    ", fast_result.min_eig)

solve_s speedup: 4.44x
baseline FDP:    0.091
fast FDP:        0.062
baseline power:  1.0
fast power:      1.0
baseline min eig:0.24654608383770443
fast min eig:    0.24654555346759
